In [1]:
import os

dirs = [
    "data",
    "data/SIDD",
    "data/SIDD/train",
    "data/SIDD/train/clean",
    "data/SIDD/train/noisy",
    "data/DND",
    "data/DND/benchmark_images",
    "models",
    "utils",
    "checkpoints",
    "outputs"
]

for d in dirs:
    os.makedirs(d, exist_ok=True)

print("Project folders created successfully!")

Project folders created successfully!


In [4]:
# utils/dataset.py (inside notebook cell for now)

import torch
from torch.utils.data import Dataset
from PIL import Image
import os
import random
import torchvision.transforms as T

class PairedImageDataset(Dataset):
    """
    A simple paired dataset loader.
    Expects:
        root/
           clean/
              img1.png
              img2.png
           noisy/
              img1.png
              img2.png
    """

    def __init__(self, root, patch_size=256):
        self.root = root
        self.clean_dir = os.path.join(root, "clean")
        self.noisy_dir = os.path.join(root, "noisy")

        # list all images
        self.clean_files = sorted([f for f in os.listdir(self.clean_dir) if f.lower().endswith((".png", ".jpg", ".jpeg"))])
        self.noisy_files = sorted([f for f in os.listdir(self.noisy_dir) if f.lower().endswith((".png", ".jpg", ".jpeg"))])

        assert len(self.clean_files) == len(self.noisy_files), "Clean/Noisy folder size mismatch"
        assert len(self.clean_files) > 0, "Dataset is empty!"

        self.patch_size = patch_size

        # transforms
        self.to_tensor = T.ToTensor()
        self.crop = T.RandomCrop(patch_size)

    def __len__(self):
        return len(self.clean_files)

    def __getitem__(self, idx):
        # get file paths
        clean_path = os.path.join(self.clean_dir, self.clean_files[idx])
        noisy_path = os.path.join(self.noisy_dir, self.noisy_files[idx])

        # open images
        clean = Image.open(clean_path).convert("RGB")
        noisy = Image.open(noisy_path).convert("RGB")

        # apply SAME crop to both images
        i, j, h, w = T.RandomCrop.get_params(clean, (self.patch_size, self.patch_size))
        clean = T.functional.crop(clean, i, j, h, w)
        noisy = T.functional.crop(noisy, i, j, h, w)

        clean = self.to_tensor(clean)
        noisy = self.to_tensor(noisy)

        return noisy, clean

In [5]:
# Final MFDNet implementation for your project (32 channels, 4 FDBlocks)
# Paste this in a new cell or save as models/mfdnet.py

import torch
import torch.nn as nn
import torch.nn.functional as F

# -----------------------------
# Depthwise separable conv: efficient building block
# -----------------------------
class SepConv(nn.Module):
    def __init__(self, in_ch, out_ch, k=3, s=1, p=1):
        super().__init__()
        # depthwise conv (groups=in_ch) -> pointwise conv
        self.depthwise = nn.Conv2d(in_ch, in_ch, k, s, p, groups=in_ch, bias=False)
        self.pointwise = nn.Conv2d(in_ch, out_ch, 1, bias=False)
        self.bn = nn.BatchNorm2d(out_ch)
        self.act = nn.ReLU(inplace=True)

    def forward(self, x):
        x = self.depthwise(x)
        x = self.pointwise(x)
        x = self.bn(x)
        return self.act(x)

# -----------------------------
# Feature Distillation Block (simplified, with lightweight attention & residual)
# Each block: several cheap SepConv layers + channel gating + residual skip
# -----------------------------
class FDBlock(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.conv1 = SepConv(ch, ch)
        self.conv2 = SepConv(ch, ch)
        self.conv3 = SepConv(ch, ch)

        # tiny channel attention (lightweight)
        self.attn = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(ch, max(1, ch//4), 1),
            nn.ReLU(inplace=True),
            nn.Conv2d(max(1, ch//4), ch, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        out = self.conv1(x)
        out = self.conv2(out)
        out = self.conv3(out)
        g = self.attn(out)         # gating per-channel
        out = out * g
        return out + x             # residual connection

# -----------------------------
# Multi-scale fusion block (keeps receptive field variety while staying light)
# -----------------------------
class MultiScaleFusion(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.pool = nn.AvgPool2d(2)
        self.conv = SepConv(ch, ch)
        self.up = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False)

    def forward(self, x):
        x2 = self.pool(x)
        x2 = self.conv(x2)
        x2 = self.up(x2)
        return x + x2

# -----------------------------
# Final MFDNet (configured to your picks: base_ch=32, num_blocks=4)
# -----------------------------
class MFDNet(nn.Module):
    def __init__(self, in_ch=3, base_ch=32, num_blocks=4):
        """
        in_ch: input channels (3 for RGB)
        base_ch: number of feature channels (32 chosen)
        num_blocks: number of FDBlocks (4 chosen)
        """
        super().__init__()
        # Head: initial feature extraction
        self.head = nn.Sequential(
            nn.Conv2d(in_ch, base_ch, 3, padding=1),
            nn.ReLU(inplace=True)
        )

        # Stacked Feature Distillation Blocks
        fdbs = []
        for _ in range(num_blocks):
            fdbs.append(FDBlock(base_ch))
        self.fdb_stack = nn.Sequential(*fdbs)

        # Multi-scale fusion
        self.msf = MultiScaleFusion(base_ch)

        # Bottleneck + reconstruction (light)
        self.bottleneck = SepConv(base_ch, base_ch)
        self.reconstruct = nn.Conv2d(base_ch, in_ch, 3, padding=1)

    def forward(self, x):
        fea = self.head(x)            # extract shallow features
        fea = self.fdb_stack(fea)     # deeper features via multiple FDBlocks
        fea = self.msf(fea)           # fuse multi-scale info
        fea = self.bottleneck(fea)    # light bottleneck
        res = self.reconstruct(fea)   # predict residual (clean - noisy)
        out = torch.clamp(x + res, 0.0, 1.0)  # residual learning: noisy + predicted residual
        return out

# ---------- quick sanity check ----------
if __name__ == "__main__":
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = MFDNet(in_ch=3, base_ch=32, num_blocks=4).to(device)
    x = torch.randn(1,3,256,256).to(device)
    y = model(x)
    print("sanity output shape:", y.shape)   # should be [1,3,256,256]

sanity output shape: torch.Size([1, 3, 256, 256])


In [2]:
# Sanity test: confirm model forward-pass works
from importlib import reload
import torch
# If you saved model code in a cell, re-run that cell before this test
model = MFDNet(in_ch=3, base_ch=32, num_blocks=4)
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
x = torch.randn(1,3,128,128).to(device)
with torch.no_grad():
    y = model(x)
print("output shape:", y.shape)

output shape: torch.Size([1, 3, 128, 128])


In [6]:
# Tiny training smoke test

import torch
from torch import nn, optim
from torch.utils.data import DataLoader
from torchvision.utils import save_image

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Training on:", device)

# --- Load dataset ---------------------------------------------------------
# IMPORTANT: You MUST place at least 2 matching images in:
# data/SIDD/train/noisy/
# data/SIDD/train/clean/
# with same exact filenames.

train_path = "data/SIDD/train"
dataset = PairedImageDataset(train_path, patch_size=128)
loader = DataLoader(dataset, batch_size=2, shuffle=True)

print("Dataset size:", len(dataset))

# --- Model ---------------------------------------------------------------
model = MFDNet(in_ch=3, base_ch=32, num_blocks=4).to(device)

criterion = nn.L1Loss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

# --- One epoch smoke test -----------------------------------------------
model.train()
for i, (noisy, clean) in enumerate(loader):
    noisy = noisy.to(device)
    clean = clean.to(device)

    optimizer.zero_grad()
    output = model(noisy)
    loss = criterion(output, clean)
    loss.backward()
    optimizer.step()

    print(f"Batch {i}, loss = {loss.item():.6f}")

    # Save one example output
    save_image(noisy[0], "outputs/noisy_example.png")
    save_image(clean[0], "outputs/clean_example.png")
    save_image(output[0], "outputs/denoised_example.png")

    break  # only one batch for now

Training on: cpu
Dataset size: 2
Batch 0, loss = 0.151078


PosixPath('/Users/abdul/Documents/mp')